# Old or new checkout page: purchase rates

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Bayesian A/B Test for Two Proportions** · a worked example from the StatNex teaching guide

**The story.** An online shop redesigned its checkout page. For two weeks, chance decided which page each visitor saw: 1,200 saw the old page and 1,200 the new one. That is about 5,000 visitors a month. On the old page 54 visitors bought (4.5%); on the new page 69 bought (5.75%).

**The question.** Should the shop switch to the new checkout now, or keep testing?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

This analysis needs `pymc` and `arviz`. This cell installs them. If Colab asks you to restart the session, restart it and choose **Run all** again.

In [ ]:
%pip install -q "pymc>=6.3" "arviz>=1.3"

## The table

This cell saves the table `14_bayesian/bayes_ab_proportions.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Variable,Old checkout,New checkout
Sample size,1200.0,1200.0
Purchase rate,0.045,0.0575

## Libraries

In [ ]:
import arviz as az
import numpy as np
import pandas as pd
import pymc as pm

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
# Each column holds the sample size and the proportion, top to bottom.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
a_name, b_name = "Old checkout", "New checkout"
credible = 95 / 100

sizes, successes, typed = [], [], []
for name in (a_name, b_name):
    size, p = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    sizes.append(int(round(size)))
    successes.append(int(round(size * p)))  # the number of successes
    typed.append(p)  # the proportion as typed
sizes, successes = np.array(sizes), np.array(successes)

## The prior, shared by A and B

In [ ]:
# Uniform, Beta(1, 1): every proportion is as likely as any other before the data.
a0, b0 = 1.0, 1.0

## Posteriors of A and B: PyMC

In [ ]:
# Each version's successes are binomial with its own rate, and the two rates share the
# prior. A beta prior on a binomial count gives a beta posterior, so each rate's update
# is two sums; PyMC samples both by MCMC (NUTS), and the difference and the lift are
# worked out draw by draw. With 100,000 draws the summaries agree with the exact ones to
# about two or three significant figures.
with pm.Model():
    rate = pm.Beta("rate", alpha=a0, beta=b0, shape=2)
    pm.Binomial("successes", n=sizes, p=rate, observed=successes)
    pm.Deterministic("difference", rate[1] - rate[0])
    pm.Deterministic("lift", rate[1] / rate[0] - 1)
    trace = pm.sample(draws=25_000, tune=2_000, chains=4, random_seed=2026, progressbar=False)
print(az.summary(trace, kind="stats", ci_prob=credible, ci_kind="eti"))

rates = trace.posterior["rate"].to_numpy().reshape(-1, 2)
difference = trace.posterior["difference"].to_numpy().ravel()
lift = trace.posterior["lift"].to_numpy().ravel()
low, high = (1 - credible) / 2, (1 + credible) / 2


def summary(values):
    return {"mean": values.mean(), "median": np.median(values), "sd": values.std(ddof=1),
            "lower": np.quantile(values, low), "upper": np.quantile(values, high)}


update = pd.DataFrame({name: {"n": sizes[k], "successes": successes[k], "proportion": typed[k],
                              "prior mean": a0 / (a0 + b0), "posterior alpha": a0 + successes[k],
                              "posterior beta": b0 + sizes[k] - successes[k]}
                       for k, name in enumerate((a_name, b_name))})
posterior = pd.DataFrame({a_name: summary(rates[:, 0]), b_name: summary(rates[:, 1])})
comparison = pd.DataFrame({f"Difference ({b_name} - {a_name})": summary(difference),
                           f"Relative difference ({b_name} / {a_name} - 1)": summary(lift)}).T
comparison = comparison[["mean", "median", "lower", "upper"]]
# The lift has a mean only when A's posterior alpha is above 1: E[1/p] is infinite
# otherwise, so the mean of the draws would be a number standing for nothing.
if a0 + successes[0] <= 1:
    comparison.iloc[1, 0] = np.nan
# The decision: how often B beats A across the draws, and the expected loss of each
# choice -- the average over the draws of what it gives up: nothing in a draw where it
# is the better one, the other's lead where it is not. It is in units of the proportion
# (0.001 is 0.1 percentage points).
decision = pd.Series({
    f"Probability that {b_name} beats {a_name}": np.mean(difference > 0),
    f"Probability that {a_name} beats {b_name}": np.mean(difference < 0),
    f"Expected loss if you choose {a_name}": np.mean(np.maximum(difference, 0)),
    f"Expected loss if you choose {b_name}": np.mean(np.maximum(-difference, 0)),
})

print("\nFrom prior to posterior")
print(update)
print(f"\nPosterior of each proportion ({credible:.0%} credible interval)")
print(posterior)
print("\nPosterior of the difference and the relative difference")
print(comparison)
print("\nWhich to choose")
print(decision.to_string())

## What the result says

On the old page 54 of 1,200 visitors bought (4.5%). On the new page 69 of 1,200 bought (5.75%). The posterior is what we believe about each true purchase rate after seeing the data. The posterior mean of the difference is 0.0125, or about 1.25 percentage points. Its 95% credible interval, the range that holds the true difference with 95% probability, runs from -0.005 to 0.030, so how much better the new page is remains uncertain.

The probability that the new page beats the old one is 0.92. The expected loss is the purchase rate a choice gives up on average: nothing when the choice is right, the shortfall when it is wrong. It is 0.0003 for the new page and 0.0128 for the old page.

Switching risks almost nothing and keeping the old page risks much more. The shop should switch now.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.